In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import litellm
from litellm import completion, embedding
from multiprocessing import Pool
from tenacity import retry, wait_exponential
import os
import numpy as np

In [10]:
load_dotenv(override=True)
openai = OpenAI(api_key=os.getenv("LCPP_TOKEN"), base_url=os.getenv("LCPP_BASE_URL"))
embedding_model = "qwen3-embedding-4b:q8_0:1k"
# model = "openai/glm-4.5-air:q4_k_m:65k"
model = "openai/GLM-4.5-Air(no think):q4_k_m:65k"

In [11]:
KNOWLEDGE_BASE_PATH = Path("/root/llm_engineering/week5/knowledge-base/")
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [12]:
texts = fetch_documents()

Loaded 76 documents


In [14]:
# Chunking using lite_llm
AVERAGE_CHUNK_SIZE = 500

class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk"
    )
    start: int = Field(
        description="The starting character index (0-based) of this chunk in the original document"
    )
    stop: int = Field(
        description="The ending character index (exclusive) of this chunk in the original document"
    )

class Chunks(BaseModel):
    chunks: list[Chunk]

def chunk_document_with_litellm(document, model=model):
    """Chunk a document using lite_llm's completion function"""
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    document_text = document["text"]
    document_length = len(document_text)
    
    prompt = f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is of type: {document.get("type", "unknown")}
The document has been retrieved from: {document.get("source", "unknown")}
The document is {document_length} characters long.

A chatbot will use these chunks to answer questions.
You should divide up the document as you see fit, being sure that the entire document is returned across the chunks - don't leave anything out.
This document should probably be split into at least {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words. A chunk must be less than 1000 tokens.

For each chunk, you should provide:
1. A headline (a brief heading for this chunk)
2. A summary (a few sentences summarizing the content)
3. A start index (the starting character position in the document, 0-based)
4. A stop index (the ending character position in the document, exclusive - this index is not included)

The start and stop indices must be valid character positions within the document (0 to {document_length - 1} for start, 1 to {document_length} for stop).
Together your chunks should represent the entire document with overlap, covering from character 0 to character {document_length - 1}.

CRITICAL: Output ONLY the JSON structure. Do not include any reasoning, explanations, thinking, or commentary. Do not include phrases like "I'll", "Let me", "Here are", or any planning statements. Output only the raw JSON response.

Here is the document:

{document_text}

Respond with the chunks in JSON format only.
/nothink
"""
    
    messages = [{"role": "user", "content": prompt}]
    
    # Stop sequences to prevent thinking leakage
    stop_sequences = [
        "\n\nI'll",
        "\n\nLet me",
        "\n\nHere are",
        "\n\nNow let me",
        "\n\nI'll create",
        "\n\nLet me create",
        "<think>",
        "transcription errors",
    ]
    
    # Use lite_llm completion with structured output
    response = completion(
        model=model,
        messages=messages,
        response_format=Chunks,
        stream=False,
        api_base=os.getenv("LCPP_BASE_URL"),
        api_key=os.getenv("LCPP_TOKEN"),
        temperature=0.1,  # Lower temperature for more deterministic output
        stop=stop_sequences,  # Stop if thinking patterns appear
    )
    
    reply = response.choices[0].message.content  # type: ignore[attr-defined]
    if reply is None:
        raise ValueError("Received empty response from model")
    
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    
    # Return chunks with metadata, extracting text using start/stop indices
    chunked_results = []
    for chunk in doc_as_chunks:
        # Validate indices
        if chunk.start < 0 or chunk.stop > document_length or chunk.start >= chunk.stop:
            raise ValueError(
                f"Invalid chunk indices: start={chunk.start}, stop={chunk.stop} "
                f"for document of length {document_length}"
            )
        
        # Extract the original text using the indices
        original_text = document_text[chunk.start:chunk.stop]
        
        chunked_results.append({
            "headline": chunk.headline,
            "summary": chunk.summary,
            "text": original_text,
            "full_content": f"{chunk.headline}\n\n{chunk.summary}\n\n{original_text}",
            "metadata": {
                "source": document.get("source", "unknown"),
                "type": document.get("type", "unknown"),
                "start": chunk.start,
                "stop": chunk.stop
            }
        })
    
    return chunked_results

# Example: Chunk the first document
if len(texts) > 0:
    first_doc = texts[0]
    chunks = chunk_document_with_litellm(first_doc)
    print(f"Created {len(chunks)} chunks from first document")
    print(f"First chunk headline: {chunks[0]['headline']}")


Created 7 chunks from first document
First chunk headline: Contract Overview


In [16]:
# Option 1: Chunk all documents using lite_llm, then create embeddings
all_chunks = []
for doc in tqdm(texts[:3]):  # Process first 3 documents as example
    chunks = chunk_document_with_litellm(doc)
    all_chunks.extend(chunks)

# Extract text content from chunked documents
text_contents = [chunk["full_content"] for chunk in all_chunks]

# Create embeddings using lite_llm
embeddings_response = embedding(
    model="openai/qwen3-embedding-4b:q8_0:1k",
    api_base=os.getenv("LCPP_BASE_URL"),
    api_key=os.getenv("LCPP_TOKEN"),
    input=text_contents
)

print(f"Created embeddings for {len(text_contents)} chunks")

# Option 2: Direct embedding without chunking (original approach)
# text_contents = [doc["text"] for doc in texts]
# openai.embeddings.create(model=embedding_model, input=text_contents)

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:14<00:00,  4.90s/it]


Created embeddings for 21 chunks


In [ ]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [ ]:
chunks = create_chunks(documents)

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")